# here we will create 3 gold table as per standard practice but we also need to calculate shrinkage % and utilization rate so we will create one extra unified table fact_shift_shrinakge_daily --for easy calculation in power BI

In [0]:
# ==============================================================================
# NOTEBOOK: 02_gold_facts
# OBJECTIVE: Materialize 3 Atomic Facts + 1 Unified Daily Workforce Fact Table
# ==============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, TimestampType, DateType

silver_schema = "7trents_data_platform.silver"
gold_schema = "7trents_data_platform.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

In [0]:


# ------------------------------------------------------------------------------
# 1. ATOMIC FACT 1: GOLD.FACT_SHIFT
# ------------------------------------------------------------------------------
print("\n--- Materializing: gold.fact_shift ---")
df_silver_shift = spark.table(f"{silver_schema}.fact_shift")

df_gold_shift = (
    df_silver_shift
    .withColumn("shift_id", F.col("shift_id").cast(IntegerType()))
    .withColumn("people_id", F.col("people_id").cast(IntegerType()))
    .withColumn("county_id", F.col("county_id").cast(IntegerType()))
    .withColumn("shift_date", F.to_date(F.col("scheduled_start")))
    .withColumn("shift_date_key", F.date_format(F.col("shift_date"), "yyyyMMdd").cast(IntegerType()))
    .withColumn("scheduled_start", F.col("scheduled_start").cast(TimestampType()))
    .withColumn("scheduled_end", F.col("scheduled_end").cast(TimestampType()))
    .withColumn(
        "shift_hours",
        F.round((F.col("scheduled_end").cast("long") - F.col("scheduled_start").cast("long")) / 3600.0, 2)
    )
    .withColumn("_gold_processed_at", F.current_timestamp())
    .select(
        "shift_id",
        "shift_date_key",
        "shift_date",
        "people_id",
        "county_id",
        "scheduled_start",
        "scheduled_end",
        "shift_hours",
        "_gold_processed_at"
    )
)

(
    df_gold_shift.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.fact_shift")
)
print(f"✓ Saved {gold_schema}.fact_shift: {df_gold_shift.count()} rows")


In [0]:


# ------------------------------------------------------------------------------
# 2. ATOMIC FACT 2: GOLD.FACT_SHRINKAGE
# ------------------------------------------------------------------------------
print("\n--- Materializing: gold.fact_shrinkage ---")
df_silver_shrinkage = spark.table(f"{silver_schema}.fact_shrinkage")

df_gold_shrinkage = (
    df_silver_shrinkage
    .withColumn("shrinkage_id", F.col("shrinkage_id").cast(IntegerType()))
    .withColumn("people_id", F.col("people_id").cast(IntegerType()))
    .withColumn("shrinkage_type_id", F.col("shrinkage_type_id").cast(IntegerType()))
    .withColumn("shrinkage_date", F.to_date(F.col("start_datetime")))
    .withColumn("shrinkage_date_key", F.date_format(F.col("shrinkage_date"), "yyyyMMdd").cast(IntegerType()))
    .withColumn("start_datetime", F.col("start_datetime").cast(TimestampType()))
    .withColumn("end_datetime", F.col("end_datetime").cast(TimestampType()))
    .withColumn(
        "shrinkage_hours",
        F.round((F.col("end_datetime").cast("long") - F.col("start_datetime").cast("long")) / 3600.0, 2)
    )
    .withColumn("_gold_processed_at", F.current_timestamp())
    .select(
        "shrinkage_id",
        "shrinkage_date_key",
        "shrinkage_date",
        "people_id",
        "shrinkage_type_id",
        "start_datetime",
        "end_datetime",
        "shrinkage_hours",
        "_gold_processed_at"
    )
)

(
    df_gold_shrinkage.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.fact_shrinkage")
)
print(f"✓ Saved {gold_schema}.fact_shrinkage: {df_gold_shrinkage.count()} rows")


In [0]:


# ------------------------------------------------------------------------------
# 3. ATOMIC FACT 3: GOLD.FACT_WORK_SNAPSHOT
# ------------------------------------------------------------------------------
print("\n--- Materializing: gold.fact_work_snapshot ---")
df_silver_work_snap = spark.table(f"{silver_schema}.fact_work_snapshot")

df_gold_work_snap = (
    df_silver_work_snap
    .withColumn("work_id", F.col("work_id").cast(IntegerType()))
    .withColumn("people_id", F.col("people_id").cast(IntegerType()))
    .withColumn("county_id", F.col("county_id").cast(IntegerType()))
    .withColumn("cancellation_id", F.col("cancellation_id").cast(IntegerType()))
    .withColumn("status", F.trim(F.col("status")))
    .withColumn("snapshot_date", F.to_date(F.col("snapshot_date")))
    .withColumn("snapshot_date_key", F.date_format(F.col("snapshot_date"), "yyyyMMdd").cast(IntegerType()))
    .withColumn("work_date", F.to_date(F.col("work_date")))
    .withColumn("work_date_key", F.date_format(F.col("work_date"), "yyyyMMdd").cast(IntegerType()))
    .withColumn("scheduled_start", F.col("scheduled_start").cast(TimestampType()))
    .withColumn("scheduled_end", F.col("scheduled_end").cast(TimestampType()))
    .withColumn(
        "planned_hours",
        F.round((F.col("scheduled_end").cast("long") - F.col("scheduled_start").cast("long")) / 3600.0, 2)
    )
    .withColumn("_gold_processed_at", F.current_timestamp())
    .select(
        "snapshot_date_key",
        "snapshot_date",
        "work_date_key",
        "work_date",
        "work_id",
        "people_id",
        "county_id",
        "status",
        "cancellation_id",
        "scheduled_start",
        "scheduled_end",
        "planned_hours",
        "_gold_processed_at"
    )
)

(
    df_gold_work_snap.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.fact_work_snapshot")
)
print(f"✓ Saved {gold_schema}.fact_work_snapshot: {df_gold_work_snap.count()} rows")


In [0]:
# ------------------------------------------------------------------------------
# 4. UNIFIED FACT 4: GOLD.FACT_SHIFT_SHRINKAGE_DAILY (Fixed: Ambiguous Key Resolved)
# ------------------------------------------------------------------------------
print("\n--- Materializing: gold.fact_shift_shrinkage_daily ---")

# A. Aggregate shifts to operative-day grain
df_shift_daily = (
    df_gold_shift
    .groupBy("people_id", "shift_date", "county_id")
    .agg(F.round(F.sum("shift_hours"), 2).alias("total_shift_hours"))
)

# B. Join shrinkage with dim_shrinkage_type to separate Sickness from Other Shrinkage
df_shrinkage_with_dim = (
    df_gold_shrinkage
    .join(spark.table(f"{silver_schema}.dim_shrinkage_type"), on="shrinkage_type_id", how="left")
)

# Rename people_id and date in shrinkage aggregate to avoid duplicate column collision
df_shrink_daily = (
    df_shrinkage_with_dim
    .groupBy(
        F.col("people_id").alias("shrink_people_id"),
        F.col("shrinkage_date").alias("shrink_date")
    )
    .agg(
        F.round(F.sum("shrinkage_hours"), 2).alias("total_shrinkage_hours"),
        F.round(
            F.sum(F.when(F.lower(F.col("shrinkage_type")) == "sickness", F.col("shrinkage_hours")).otherwise(0.0)),
            2
        ).alias("sickness_hours"),
        F.round(
            F.sum(F.when(F.lower(F.col("shrinkage_type")) != "sickness", F.col("shrinkage_hours")).otherwise(0.0)),
            2
        ).alias("other_shrinkage_hours")
    )
)

# C. Full outer join using distinct column names
df_unified_daily = (
    df_shift_daily
    .join(
        df_shrink_daily,
        (df_shift_daily["people_id"] == df_shrink_daily["shrink_people_id"]) & 
        (df_shift_daily["shift_date"] == df_shrink_daily["shrink_date"]),
        how="full_outer"
    )
    .withColumn("resolved_people_id", F.coalesce(F.col("people_id"), F.col("shrink_people_id")).cast(IntegerType()))
    .withColumn("capacity_date", F.coalesce(F.col("shift_date"), F.col("shrink_date")))
    .withColumn("date_key", F.date_format(F.col("capacity_date"), "yyyyMMdd").cast(IntegerType()))
    .withColumn("total_shift_hours", F.coalesce(F.col("total_shift_hours"), F.lit(0.0)))
    .withColumn("total_shrinkage_hours", F.coalesce(F.col("total_shrinkage_hours"), F.lit(0.0)))
    .withColumn("sickness_hours", F.coalesce(F.col("sickness_hours"), F.lit(0.0)))
    .withColumn("other_shrinkage_hours", F.coalesce(F.col("other_shrinkage_hours"), F.lit(0.0)))
    .withColumn("_gold_processed_at", F.current_timestamp())
    .select(
        "date_key",
        "capacity_date",
        F.col("resolved_people_id").alias("people_id"),
        "county_id",
        "total_shift_hours",
        "total_shrinkage_hours",
        "sickness_hours",
        "other_shrinkage_hours",
        "_gold_processed_at"
    )
)

(
    df_unified_daily.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.fact_shift_shrinkage_daily")
)
print(f"✓ Saved {gold_schema}.fact_shift_shrinkage_daily: {df_unified_daily.count()} rows")

In [0]:

# ------------------------------------------------------------------------------
# 5. VERIFICATION AUDIT
# ------------------------------------------------------------------------------
print("\n" + "="*50)
print("ALL 4 GOLD FACT TABLES VERIFIED:")
print("="*50)

fact_tables = [
    "fact_shift",
    "fact_shrinkage",
    "fact_work_snapshot",
    "fact_shift_shrinkage_daily"
]
for tbl in fact_tables:
    total = spark.table(f"{gold_schema}.{tbl}").count()
    print(f"Table: {gold_schema}.{tbl:<30} Total Rows: {total}")